In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.6 MB/s eta 0:00:00


In [2]:
from Bio import Entrez
from time import sleep
import pandas as pd
from tqdm import tqdm
import json

SEARCH_TERM = '"creatine supplementation"'
OUTPUT_FILE = "pubmed_creatine.csv"
BATCH_SIZE = 100

Entrez.email = "nicolas.forcella@ru.nl"

handle_search = Entrez.esearch(
    db="pubmed",
    term=SEARCH_TERM,
    retmax=500000
)

record = Entrez.read(handle_search)
handle_search.close()

pubmed_ids = record["IdList"]
print(f"found {len(pubmed_ids)} papers.")

found 1092 papers.


In [3]:
#Fetch all publications in batches
def fetch_publications(pubmed_ids):
    ids = ",".join(pubmed_ids) #Convert Ids to string
    sleep(0.3)  # sleep for avoiding too many requests
    handle_records = Entrez.efetch(
        db="pubmed",
        id=ids,
        rettype="medline",
        retmode="xml"
    )
    records = Entrez.read(handle_records)
    handle_records.close()
    return records


#Get abstract date year and publication type
global all_pub_types
all_pub_types = set() #store a set of all types retrieved
def parse_publication(article):

    citation = article.get("MedlineCitation", {})
    article_data = citation.get("Article", {})

    # Title
    title = article_data.get("ArticleTitle", "")

    # Abstract
    abstract = ""
    abs_data = article_data.get("Abstract")
    if abs_data and "AbstractText" in abs_data:
        abstract = " ".join([str(text) for text in abs_data["AbstractText"]])

    # Year
    year = None
    journal_date = article_data.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
    year = journal_date.get("Year", None)

    if not year:
      medline_date = journal_date.get("MedlineDate", None)
      year = medline_date.strip().split(' ')[0] #Get year

    # Journal
    journal = article_data.get("Journal", {}).get("Title", "")

    # Article Types
    pub_types_list = article_data.get("PublicationTypeList", [])
    pub_types = [str(publication_type) for publication_type in pub_types_list]
    all_pub_types.update(pub_types)

    # PubmedId
    pubmed_id = citation.get("PMID", "")

    return {
        "pmid": pubmed_id,
        "title": title,
        "abstract": abstract,
        "year": int(year),
        "journal": journal,
        "article_type": "; ".join(pub_types)
    }


In [4]:
all_articles_raw = []

for start_batch in tqdm(range(0, len(pubmed_ids), BATCH_SIZE)):
  end_batch = start_batch + BATCH_SIZE
  batch_publication_ids = pubmed_ids[start_batch:start_batch + BATCH_SIZE] #Get next publications to fetch
  data = fetch_publications(batch_publication_ids)

  for article in data["PubmedArticle"]:
      all_articles_raw.append(article)

100%|██████████| 11/11 [00:27<00:00,  2.45s/it]


In [5]:
all_articles_clean = []
for article_raw in all_articles_raw:
  all_articles_clean.append(parse_publication(article_raw))


In [6]:
all_pub_types

{'Case Reports',
 'Clinical Study',
 'Clinical Trial',
 'Clinical Trial Protocol',
 'Clinical Trial, Veterinary',
 'Comment',
 'Comparative Study',
 'Controlled Clinical Trial',
 'Editorial',
 'English Abstract',
 'Evaluation Study',
 'Historical Article',
 'Journal Article',
 'Letter',
 'Meta-Analysis',
 'Multicenter Study',
 'Observational Study',
 'Preprint',
 'Published Erratum',
 'Randomized Controlled Trial',
 'Research Support, N.I.H., Extramural',
 "Research Support, Non-U.S. Gov't",
 "Research Support, U.S. Gov't, Non-P.H.S.",
 "Research Support, U.S. Gov't, P.H.S.",
 'Review',
 'Scoping Review',
 'Systematic Review',
 'Validation Study'}

# 1. Filter data

In [7]:
df = pd.DataFrame(all_articles_clean)

In [8]:
df['year'] = df['year'].astype(int)
#Drop rows were abstract, title or year info is not present
df = df.dropna(subset=["abstract", "title", "year"])
#Filter non important types
TYPES_TO_FILTER = ['Comment', 'Published Erratum']
def filter_irrelevant_types(types):
  if any(t in types for t in TYPES_TO_FILTER):
    return False
  return True

df = df[df['article_type'].apply(filter_irrelevant_types)]
df["raw_corpus"] = df["title"] + " " + df["abstract"]

In [9]:
print(f"A total of {len(df)} articles are left")

A total of 1071 articles are left


In [10]:
ALL_TYPES = {'Case Reports',
 'Clinical Study',
 'Clinical Trial',
 'Clinical Trial Protocol',
 'Clinical Trial, Veterinary',
 'Comparative Study',
 'Controlled Clinical Trial',
 'Editorial',
 'English Abstract',
 'Evaluation Study',
 'Historical Article',
 'Journal Article',
 'Meta-Analysis',
 'Multicenter Study',
 'Preprint',
 'Randomized Controlled Trial',
 'Research Support, N.I.H., Extramural',
 "Research Support, Non-U.S. Gov't",
 "Research Support, U.S. Gov't, Non-P.H.S.",
 "Research Support, U.S. Gov't, P.H.S.",
 'Review',
 'Scoping Review',
 'Systematic Review',
 'Validation Study'}

REVIEW_TYPES = [
    "Review",
    "Systematic Review",
    "Scoping Review",
    "Meta-Analysis",
    "Comment",
    "Editorial",
    "Published Erratum",
    "English Abstract"
    'Historical Article',
]

def map_pubmed_type(pt):
    if any(t in pt for t in REVIEW_TYPES):
        return "review"
    return "research"

df['document_category'] = df['article_type'].apply(map_pubmed_type)

In [11]:
df_research = df[df['document_category'] == 'research']
df_research = df_research.reset_index(drop=True)
df = df_research
print(f"Number of research articles: {len(df_research)}")

Number of research articles: 806


In [12]:
df.to_csv(OUTPUT_FILE, index=False)